# JEV reference-equivalence audit

The first cells analyze archived judgments offline. Judge scores are not established as calibrated probabilities or independent legal correctness. Archived judgments used an incorrect human-expert-reference description; corrected `reference-v2` runs use a separate directory and input fingerprints. Human labels are adjudicated reference-equivalence labels. `Strict` has an undocumented separate rubric.

The optional final cells make API calls and require `typesafe-sdk` and credentials; they have not been rerun for the paper.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
while not (ROOT / "scripts" / "jev").is_dir():
    if ROOT == ROOT.parent:
        raise RuntimeError("Run from inside LEGAL-UQA")
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import pandas as pd
from scripts.jev.pipeline import OUT_DIR, REVISED_OUT_DIR, HUMAN_EVAL_FILES, run_model
from scripts.jev.agreement import summarise

## Archived agreement

Nine models answer the same 124 questions: 1,116 judgments are repeated observations, not independent questions. Compare agreement with the always-positive baseline and kappa. Source CSV human-label copies reflect the three logged corrections; original judge scores are preserved.

In [ ]:
summary = pd.DataFrame([summarise(p) for p in sorted(OUT_DIR.glob("*_jev_verdicts.csv"))])
summary.sort_values("accuracy", ascending=False)

In [ ]:
model_key = "claude-sonnet-5"
df = pd.read_csv(OUT_DIR / f"{model_key}_jev_verdicts.csv")
disagree = df.loc[df.human_score.ge(0.5) != df.noul.ge(0.5)].sort_values("noul")
disagree[["id", "question", "ground_truth", "prediction", "human_score", "noul"]]

## Optional new-prompt inference

Running the next cells makes API requests. The corrected prompt names the model-generated translated reference and requires complete equivalence. Outputs are written under `reference-v2/`; they do not replace the paper’s archived experiment. Top-level `await` works in Jupyter’s existing event loop. No performance or runtime estimate is asserted.

In [ ]:
csv_path = await run_model("claude-3.5-sonnet", concurrency=4, limit=1)
csv_path

In [ ]:
for key in HUMAN_EVAL_FILES:
    await run_model(key, concurrency=8)

In [ ]:
new_paths = sorted(REVISED_OUT_DIR.glob("*_jev_verdicts.csv"))
pd.DataFrame([summarise(p) for p in new_paths])